# Parte 3 — Ecuaciones elípticas
## 3.3 El principio del máximo
### 3.3.02 Comparación, subsoluciones, supersoluciones y términos de orden cero

**Fuente principal:** página 7 de las notas manuscritas del archivo
`Ecuación De Onda(1).pdf`.

Este notebook continúa exactamente después de
`03.3.01_Principios_del_maximo_unicidad_estabilidad.ipynb`.

Se trabajan únicamente tres bloques coherentes:

1. principio de comparación para funciones armónicas;
2. subsoluciones y supersoluciones;
3. operadores de la forma $\Delta u+h(x)u$ y la condición de signo $h\leq0$.

El teorema de diferenciación de Lebesgue, que aparece al final de la página,
queda para una unidad posterior.

## Convenciones editoriales

- **Transcripción literal de las notas:** conserva el orden y la notación visible.
- **Aclaración:** explicita una hipótesis necesaria que no está escrita completamente.
- **Corrección editorial:** corrige un signo o una formulación ambigua sin alterar el original.
- **Complemento:** añade una demostración o consecuencia necesaria para el Examen General.
- Toda la matemática usa solamente `$...$` y `$$...$$`.

## Página original de las notas

![Página 7 — comparación y sub/supersoluciones](../fuentes/pagina_07_notas.png)

En la página se distinguen tres bloques: el teorema de comparación,
la definición de subsolución para $\Delta u+h(x)u=f$ y el principio débil
del máximo bajo la condición $h\leq0$.

# Simulaciones y gráficas

Las celdas siguientes se ejecutan directamente. No existe una bandera
`VIDEO=True` que deba modificarse.

El notebook genera:

1. una animación de la diferencia entre dos extensiones armónicas con datos ordenados;
2. barreras inferior y superior para un problema con $h<0$;
3. un contraejemplo cuando $h>0$;
4. un barrido del parámetro de orden cero cerca del primer valor propio.

CuPy/CUDA se usa automáticamente en la relajación bidimensional cuando está disponible.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    """Convierte un arreglo CuPy a NumPy; deja intacto un arreglo NumPy."""
    if GPU_AVAILABLE:
        return cp.asnumpy(array)
    return np.asarray(array)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=140,
    bitrate=9000,
):
    """
    Guarda y muestra automáticamente una animación.

    Se usa MP4 si FFmpeg está disponible. En otro caso se genera un GIF.
    """
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(output, writer=writer, dpi=min(dpi, 110))
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output

## Simulación 3.3.A — Comparación de extensiones armónicas

Sean $u_1$ y $u_2$ armónicas en el cuadrado. La diferencia

$$
w=u_1-u_2
$$

también es armónica. Se prescriben datos de frontera para los cuales

$$
w\geq0
\qquad\text{sobre }\partial\Omega,
$$

y $w$ no es idénticamente cero en la frontera.

La animación muestra la relajación hacia la extensión armónica de esos datos.
El principio de comparación predice

$$
w>0
\qquad\text{en }\Omega.
$$

In [ ]:
# ============================================================
# COMPARACIÓN ARMÓNICA EN UN CUADRADO
# ============================================================

N = 320 if GPU_AVAILABLE else 170
N_ITER = 4200 if GPU_AVAILABLE else 2300
N_FRAMES = 180 if GPU_AVAILABLE else 120
FPS = 120 if GPU_AVAILABLE else 60

x = xp.linspace(0.0, 1.0, N)
y = xp.linspace(0.0, 1.0, N)
X, Y = xp.meshgrid(x, y, indexing="xy")

W = xp.zeros((N, N), dtype=xp.float32)

# La diferencia de los datos es no negativa.
top_difference = xp.sin(math.pi * x) ** 2
right_difference = 0.25 * xp.sin(math.pi * y) ** 2

W[-1, :] = top_difference
W[:, -1] = xp.maximum(W[:, -1], right_difference)

save_steps = set(
    np.unique(
        np.round(
            np.geomspace(1, N_ITER, N_FRAMES)
        ).astype(int)
    ).tolist()
)

snapshots = [to_cpu(W).copy()]
iterations = [0]
interior_minima = [float(to_cpu(xp.min(W[1:-1, 1:-1])))]
interior_maxima = [float(to_cpu(xp.max(W[1:-1, 1:-1])))]

omega = 0.88

for iteration in range(1, N_ITER + 1):
    average = 0.25 * (
        W[2:, 1:-1]
        + W[:-2, 1:-1]
        + W[1:-1, 2:]
        + W[1:-1, :-2]
    )

    W_new = W.copy()
    W_new[1:-1, 1:-1] = (
        (1.0 - omega) * W[1:-1, 1:-1]
        + omega * average
    )

    # Reimposición de los datos.
    W_new[0, :] = 0.0
    W_new[:, 0] = 0.0
    W_new[-1, :] = top_difference
    W_new[:, -1] = xp.maximum(
        W_new[:, -1],
        right_difference,
    )

    W = W_new

    if iteration in save_steps:
        snapshots.append(to_cpu(W).copy())
        iterations.append(iteration)
        interior_minima.append(
            float(to_cpu(xp.min(W[1:-1, 1:-1])))
        )
        interior_maxima.append(
            float(to_cpu(xp.max(W[1:-1, 1:-1])))
        )

snapshots = np.asarray(snapshots)
iterations = np.asarray(iterations)
interior_minima = np.asarray(interior_minima)
interior_maxima = np.asarray(interior_maxima)

print("Mínimo interior final:", interior_minima[-1])
print("Máximo interior final:", interior_maxima[-1])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

image = ax.imshow(
    snapshots[0],
    origin="lower",
    extent=[0.0, 1.0, 0.0, 1.0],
    interpolation="bilinear",
    vmin=0.0,
    vmax=float(np.max(snapshots[-1])),
)
fig.colorbar(image, ax=ax, label=r"$w=u_1-u_2$")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Relajación de la diferencia armónica")

status = ax.text(
    0.02,
    0.98,
    "",
    transform=ax.transAxes,
    va="top",
    bbox={"boxstyle": "round", "alpha": 0.8},
)


def update_comparison(frame):
    image.set_data(snapshots[frame])
    status.set_text(
        f"iteración = {iterations[frame]}\n"
        + rf"$\min_{{\Omega_h^\circ}}w={interior_minima[frame]:.3e}$"
        + "\n"
        + rf"$\max_{{\Omega_h^\circ}}w={interior_maxima[frame]:.3e}$"
    )
    return image, status


animation = FuncAnimation(
    fig,
    update_comparison,
    frames=len(snapshots),
    interval=1000.0 / FPS,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.3.A_comparacion_armonica",
    fps=FPS,
    dpi=180 if GPU_AVAILABLE else 140,
    bitrate=18000 if GPU_AVAILABLE else 9000,
)

plt.close(fig)

In [ ]:
# Perfil interior de la diferencia final.

W_final = snapshots[-1]
mid = len(y) // 2

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(to_cpu(x), W_final[mid, :])
ax.axhline(0.0, linewidth=1.0)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$w(x,1/2)$")
ax.set_title("La diferencia permanece positiva en el interior")
ax.grid(True, alpha=0.3)
fig.tight_layout()

path = FIG_DIR / "03.3.A_perfil_comparacion.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.3.B — Barreras inferior y superior

Consideramos el operador

$$
L_hv=\Delta v+h\,v,
\qquad
h=-3,
$$

en $\Omega=(-1,1)^2$.

La solución de referencia es

$$
u(x,y)
=
\sin\left(\frac{\pi(x+1)}{2}\right)
\sin\left(\frac{\pi(y+1)}{2}\right).
$$

Se define

$$
\psi(x,y)=(1-x^2)(1-y^2)
$$

y, para $\varepsilon>0$,

$$
\underline u_\varepsilon=u-\varepsilon\psi,
\qquad
\overline u_\varepsilon=u+\varepsilon\psi.
$$

Como $L_h\psi\leq0$ en $\overline\Omega$,

$$
L_h\underline u_\varepsilon\geq L_hu,
\qquad
L_h\overline u_\varepsilon\leq L_hu.
$$

Con la convención de las notas, $\underline u_\varepsilon$ es una subsolución y
$\overline u_\varepsilon$ una supersolución.

In [ ]:
# ============================================================
# BARRERAS PARA Delta u + h u = f
# ============================================================

n_plot = 301
xx = np.linspace(-1.0, 1.0, n_plot)
yy = np.linspace(-1.0, 1.0, n_plot)
XX, YY = np.meshgrid(xx, yy, indexing="xy")

h_value = -3.0

u_exact = (
    np.sin(0.5 * math.pi * (XX + 1.0))
    * np.sin(0.5 * math.pi * (YY + 1.0))
)

psi = (1.0 - XX**2) * (1.0 - YY**2)

# Delta u = -(pi^2/2) u.
f_value = -(0.5 * math.pi**2 + 3.0) * u_exact

# Delta psi + h psi.
L_psi = (
    -4.0
    + 2.0 * (XX**2 + YY**2)
    + h_value * psi
)

epsilon = 0.40
subsolution = u_exact - epsilon * psi
supersolution = u_exact + epsilon * psi

residual_sub = -epsilon * L_psi
residual_super = epsilon * L_psi

print("mínimo de L(sub)-f:", np.min(residual_sub))
print("máximo de L(super)-f:", np.max(residual_super))
print("mínimo de u-sub:", np.min(u_exact - subsolution))
print("mínimo de super-u:", np.min(supersolution - u_exact))

In [ ]:
# Mapa del encajonamiento por barreras.

mid = n_plot // 2

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(xx, subsolution[mid, :], label=r"$\underline u$")
ax.plot(xx, u_exact[mid, :], label=r"$u$")
ax.plot(xx, supersolution[mid, :], label=r"$\overline u$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"valor sobre $y=0$")
ax.set_title("Barreras inferior y superior")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.3.B_barreras_corte.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

In [ ]:
# Animación automática: crecimiento de la separación entre las barreras.

epsilon_frames = np.linspace(0.0, 0.65, 150)

fig, ax = plt.subplots(figsize=(10, 6))
line_sub, = ax.plot([], [], label=r"$\underline u_\varepsilon$")
line_exact, = ax.plot(xx, u_exact[mid, :], label=r"$u$")
line_super, = ax.plot([], [], label=r"$\overline u_\varepsilon$")

ax.set_xlim(-1.0, 1.0)
ax.set_ylim(
    float(np.min(u_exact[mid, :] - 0.70 * psi[mid, :])) - 0.05,
    float(np.max(u_exact[mid, :] + 0.70 * psi[mid, :])) + 0.05,
)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"valor sobre $y=0$")
ax.set_title("Familia de barreras")
ax.grid(True, alpha=0.3)
ax.legend()

status = ax.text(
    0.02,
    0.96,
    "",
    transform=ax.transAxes,
    va="top",
)


def update_barriers(frame):
    eps = epsilon_frames[frame]
    lower = u_exact[mid, :] - eps * psi[mid, :]
    upper = u_exact[mid, :] + eps * psi[mid, :]

    line_sub.set_data(xx, lower)
    line_super.set_data(xx, upper)
    status.set_text(rf"$\varepsilon={eps:.3f}$")

    return line_sub, line_exact, line_super, status


animation_barriers = FuncAnimation(
    fig,
    update_barriers,
    frames=len(epsilon_frames),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation_barriers,
    "03.3.B_familia_de_barreras",
    fps=60,
    dpi=150,
    bitrate=10000,
)

plt.close(fig)

## Simulación 3.3.C — Qué falla cuando $h>0$

En el intervalo $(0,\pi)$, la función

$$
v(x)=\sin x
$$

satisface

$$
v''+v=0,
\qquad
v(0)=v(\pi)=0,
$$

pero

$$
v(x)>0
\qquad
\text{para }0<x<\pi.
$$

Por tanto, el principio débil del máximo y la unicidad con dato de Dirichlet cero
pueden fallar cuando el coeficiente de orden cero tiene signo positivo.

También se estudia

$$
u''+\lambda u=1,
\qquad
u(0)=u(\pi)=0,
$$

cuyas soluciones crecen cerca de $\lambda=1$, el primer valor propio de $-\frac{d^2}{dx^2}$.

In [ ]:
# Contraejemplo elemental.

x1 = np.linspace(0.0, math.pi, 1000)
v = np.sin(x1)
Lv = -np.sin(x1) + np.sin(x1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x1, v, label=r"$v(x)=\sin x$")
ax.plot(x1, Lv, linestyle="--", label=r"$v''+v$")
ax.axhline(0.0, linewidth=1.0)
ax.set_xlabel(r"$x$")
ax.set_title(r"Fallo del principio del máximo para $h=1$")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.3.C_contraejemplo_h_positivo.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

In [ ]:
# Barrido del parámetro lambda cerca del primer valor propio.

def exact_solution_parameter(x, lam):
    """
    Solución de u'' + lam*u = 1, u(0)=u(pi)=0,
    para lam>0 y fuera de los valores propios k^2.
    """
    k = math.sqrt(lam)
    denominator = lam * math.sin(k * math.pi)

    if abs(denominator) < 1e-10:
        return np.full_like(x, np.nan)

    coefficient = (
        math.cos(k * math.pi) - 1.0
    ) / denominator

    return (
        coefficient * np.sin(k * x)
        - np.cos(k * x) / lam
        + 1.0 / lam
    )


lambda_left = np.linspace(0.08, 0.985, 180)
lambda_right = np.linspace(1.015, 1.85, 170)
lambda_values = np.concatenate([lambda_left, lambda_right])

max_norms = []

for lam in lambda_values:
    solution = exact_solution_parameter(x1, float(lam))
    max_norms.append(np.nanmax(np.abs(solution)))

max_norms = np.asarray(max_norms)

fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogy(lambda_values, max_norms)
ax.axvline(1.0, linestyle="--", label=r"$\lambda_1=1$")
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel(r"$\|u_\lambda\|_\infty$")
ax.set_title("Amplificación cerca del primer valor propio")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.3.C_resonancia_orden_cero.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

# 3.3.1 Principio de comparación para funciones armónicas

### **Transcripción literal de las notas**

**Teorema (Comparación).**

Sea $\Omega\subset\mathbb R^n$ abierto, acotado y conexo. Sean

$$
u_1,u_2\in C^2(\Omega)\cap C(\overline\Omega)
$$

armónicas, con datos

$$
u_i=g_i
\qquad\text{en }\partial\Omega,
\qquad
i=1,2.
$$

Si

$$
g_1\geq g_2
\qquad\text{en }\partial\Omega,
$$

entonces

$$
u_1\geq u_2
\qquad\text{en }\Omega.
$$

Si, además, los datos no son idénticos, la desigualdad es estricta en el interior.

## **Teorema 3.3.9 (Principio de comparación para funciones armónicas).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado. Sean

$$
u_1,u_2\in C^2(\Omega)\cap C(\overline\Omega)
$$

funciones armónicas en $\Omega$. Supongamos que

$$
u_1\geq u_2
\qquad\text{sobre }\partial\Omega.
$$

Entonces

$$
u_1\geq u_2
\qquad\text{en }\overline\Omega.
$$

### Demostración

Definimos

$$
w=u_1-u_2.
$$

Por linealidad,

$$
\Delta w=0
\qquad\text{en }\Omega.
$$

Además,

$$
w\geq0
\qquad\text{sobre }\partial\Omega.
$$

El principio débil del mínimo para funciones armónicas implica

$$
\min_{\overline\Omega}w
=
\min_{\partial\Omega}w
\geq0.
$$

Por tanto, $w\geq0$ en $\overline\Omega$, es decir,

$$
u_1\geq u_2.
$$

$\square$

## **Corolario 3.3.10 (Principio de comparación estricta).**

Bajo las hipótesis del Teorema 3.3.9, supongamos además que

$$
u_1-u_2
$$

no es idénticamente cero. Entonces

$$
u_1>u_2
\qquad\text{en }\Omega.
$$

### Demostración

La función $w=u_1-u_2$ es armónica y no negativa. Si existiera
$x_0\in\Omega$ con $w(x_0)=0$, entonces $w$ alcanzaría un mínimo interior.
El principio fuerte del mínimo implicaría que $w$ es constante. Como $w$
no es idénticamente cero, esto es imposible.

Por tanto,

$$
w>0
\qquad\text{en }\Omega.
$$

$\square$

### **Aclaración.**

La condición correcta para obtener desigualdad estricta es que $u_1-u_2$ no sea
idénticamente cero. Basta, por ejemplo, que $g_1\geq g_2$ en toda la frontera y que

$$
g_1(x_*)>g_2(x_*)
$$

en algún punto $x_*\in\partial\Omega$.

### Ejercicios — Sección 3.3.1

1. Sean $u,v\in C^2(\Omega)\cap C(\overline\Omega)$ armónicas en un dominio
   acotado. Demuestre que

   $$
   \max_{\overline\Omega}|u-v|
   =
   \max_{\partial\Omega}|u-v|.
   $$

2. Sean $u_1,u_2$ soluciones de

   $$
   -\Delta u_i=f_i
   \qquad\text{en }\Omega,
   $$

   con $u_1\leq u_2$ sobre $\partial\Omega$. Determine una condición entre
   $f_1$ y $f_2$ que garantice $u_1\leq u_2$ en $\Omega$.

3. Demuestre que la extensión armónica depende monótonamente del dato de Dirichlet.

4. **Tipo Examen General.** Sea $\Omega\subset\mathbb R^n$ un dominio acotado y
   conexo. Sean $u,v\in C^2(\Omega)\cap C(\overline\Omega)$ armónicas.
   Suponga que $u\geq v$ sobre $\partial\Omega$ y que $u(x_0)=v(x_0)$ para
   algún $x_0\in\Omega$. Demuestre que $u\equiv v$.

# 3.3.2 Operadores con término de orden cero

### **Transcripción literal de las notas**

Se considera la ecuación

$$
\Delta u+h\,u=f
\qquad\text{en }\Omega,
$$

donde

$$
h=h(x)
$$

es el coeficiente de orden cero.

Si

$$
h=f=0,
$$

entonces $u$ es armónica en $\Omega$.

## **Definición 3.3.11 (Operador elíptico con término de orden cero).**

Sea $\Omega\subset\mathbb R^n$ abierto y sea $h\in C(\Omega)$. Definimos

$$
L_hu:=\Delta u+h(x)u.
$$

El término $h(x)u$ es de orden cero porque no contiene derivadas de $u$.

### **Observación 3.3.12 (Importancia del signo de $h$).**

La condición

$$
h\leq0
$$

permite extender el principio del máximo elemental de $\Delta$ al operador $L_h$.

Cuando $h>0$, el principio puede fallar. El contraejemplo
$v(x)=\sin x$ en $(0,\pi)$ muestra que el signo no es una formalidad técnica.

# 3.3.3 Subsoluciones y supersoluciones

### **Transcripción literal de las notas**

Sean $f,h\in C(\Omega)$ y considérese

$$
\Delta u+h\,u=f
\qquad\text{en }\Omega.
$$

Una función $v\in C^2(\Omega)$ es una subsolución si

$$
\Delta v+h\,v\geq f
\qquad\text{en }\Omega.
$$

La desigualdad contraria define una supersolución.

## **Definición 3.3.13 (Subsolución clásica).**

Sea $\Omega\subset\mathbb R^n$ abierto y sean $f,h\in C(\Omega)$.
Una función

$$
\underline u\in C^2(\Omega)
$$

se llama **subsolución clásica** de

$$
L_hu=f
$$

si

$$
L_h\underline u\geq f
\qquad\text{en }\Omega.
$$

## **Definición 3.3.14 (Supersolución clásica).**

Una función

$$
\overline u\in C^2(\Omega)
$$

se llama **supersolución clásica** si

$$
L_h\overline u\leq f
\qquad\text{en }\Omega.
$$

### **Aclaración sobre la convención de signos.**

Algunos textos usan la convención opuesta. En este notebook se conserva la
convención de las notas:

$$
\text{subsolución}\Longleftrightarrow L_h\underline u\geq f,
$$

$$
\text{supersolución}\Longleftrightarrow L_h\overline u\leq f.
$$

## **Teorema 3.3.15 (Principio débil del máximo para $L_h$).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado y sea
$h\in C(\overline\Omega)$ tal que

$$
h\leq0
\qquad\text{en }\overline\Omega.
$$

Supongamos que

$$
v\in C^2(\Omega)\cap C(\overline\Omega)
$$

satisface

$$
L_hv=\Delta v+h(x)v\geq0
\qquad\text{en }\Omega
$$

y

$$
v\leq0
\qquad\text{sobre }\partial\Omega.
$$

Entonces

$$
v\leq0
\qquad\text{en }\overline\Omega.
$$

### Demostración añadida

Como $h$ es continua en el compacto $\overline\Omega$, existe $H\geq0$ tal que

$$
-H\leq h(x)\leq0.
$$

Tras una rotación y una traslación, podemos suponer que $\Omega$ está contenida
en una franja

$$
a<x_1<b.
$$

Elegimos $\beta>0$ de modo que

$$
\beta^2>H
$$

y definimos

$$
\phi(x)=e^{\beta x_1}.
$$

Entonces

$$
L_h\phi
=
\left(\beta^2+h(x)\right)e^{\beta x_1}>0.
$$

Para $\varepsilon>0$, sea

$$
v_\varepsilon=v+\varepsilon\phi.
$$

Se tiene

$$
L_hv_\varepsilon>0.
$$

Supongamos que $v$ toma un valor positivo en $\Omega$. Para $\varepsilon$
suficientemente pequeño, $v_\varepsilon$ alcanza un máximo positivo en un punto
interior $x_\varepsilon$. En dicho punto,

$$
\Delta v_\varepsilon(x_\varepsilon)\leq0
$$

y, como $h\leq0$ y $v_\varepsilon(x_\varepsilon)>0$,

$$
h(x_\varepsilon)v_\varepsilon(x_\varepsilon)\leq0.
$$

Por tanto,

$$
L_hv_\varepsilon(x_\varepsilon)\leq0,
$$

lo cual contradice $L_hv_\varepsilon>0$.

Así, $v\leq0$ en $\Omega$.

$\square$

## **Teorema 3.3.16 (Principio de comparación para $L_h$).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado y sea
$h\in C(\overline\Omega)$ con $h\leq0$. Sean

$$
u,v\in C^2(\Omega)\cap C(\overline\Omega)
$$

tales que

$$
L_hu\geq L_hv
\qquad\text{en }\Omega
$$

y

$$
u\leq v
\qquad\text{sobre }\partial\Omega.
$$

Entonces

$$
u\leq v
\qquad\text{en }\overline\Omega.
$$

### Demostración

Definimos

$$
w=u-v.
$$

Entonces

$$
L_hw\geq0
\qquad\text{en }\Omega
$$

y

$$
w\leq0
\qquad\text{sobre }\partial\Omega.
$$

El Teorema 3.3.15 implica $w\leq0$ en $\overline\Omega$.

$\square$

## **Corolario 3.3.17 (Unicidad para el problema de Dirichlet).**

Bajo la condición $h\leq0$, el problema

$$
\begin{cases}
\Delta u+h(x)u=f, & \text{en }\Omega,\\
u=g, & \text{sobre }\partial\Omega
\end{cases}
$$

tiene a lo más una solución en
$C^2(\Omega)\cap C(\overline\Omega)$.

## **Proposición 3.3.18 (Encajonamiento mediante barreras).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado y sea
$h\in C(\overline\Omega)$ con $h\leq0$. Supongamos que

$$
\underline u,\ u,\ \overline u
\in C^2(\Omega)\cap C(\overline\Omega)
$$

satisfacen

$$
L_h\underline u\geq f,
\qquad
L_hu=f,
\qquad
L_h\overline u\leq f
$$

en $\Omega$, y

$$
\underline u\leq u\leq\overline u
\qquad\text{sobre }\partial\Omega.
$$

Entonces

$$
\underline u\leq u\leq\overline u
\qquad\text{en }\overline\Omega.
$$

### Demostración

Aplicamos el Teorema 3.3.16 primero a
$\underline u$ y $u$, y después a $u$ y $\overline u$.

$\square$

## **Contraejemplo 3.3.19 (Fallo cuando $h>0$).**

Sea

$$
\Omega=(0,\pi)
$$

y considérese

$$
L_1v=v''+v.
$$

La función

$$
v(x)=\sin x
$$

satisface

$$
L_1v=0
\qquad\text{en }(0,\pi)
$$

y

$$
v(0)=v(\pi)=0,
$$

pero $v>0$ en el interior.

Por tanto:

1. una solución con dato de Dirichlet cero no tiene por qué ser trivial;
2. el principio débil del máximo puede fallar;
3. la unicidad puede fallar;
4. la comparación puede fallar.

### **Observación 3.3.20 (Relación con los valores propios).**

El contraejemplo aparece porque $1$ es el primer valor propio de

$$
-\frac{d^2}{dx^2}
$$

en $(0,\pi)$ con condiciones de Dirichlet.

La hipótesis $h\leq0$ es una condición elemental suficiente para el principio del máximo.
La teoría más fina se formula en términos del primer valor propio principal.

### Ejercicios — Secciones 3.3.2 y 3.3.3

1. Verifique directamente que las funciones de la Simulación 3.3.B satisfacen

   $$
   L_h\underline u\geq f,
   \qquad
   L_h\overline u\leq f.
   $$

2. Sea $\Omega=B_R(0)$ y sea

   $$
   L_hu=\Delta u-\mu^2u,
   \qquad
   \mu>0.
   $$

   Construya una barrera radial para una solución de $L_hu=f$ con dato de
   Dirichlet acotado.

3. Demuestre que si $h\leq0$, $f\geq0$ y

   $$
   \Delta u+h u=f,
   \qquad
   u\leq0\text{ sobre }\partial\Omega,
   $$

   entonces $u\leq0$ en $\Omega$.

4. Encuentre todas las soluciones de

   $$
   u''+\lambda u=0,
   \qquad
   u(0)=u(\pi)=0,
   $$

   y determine para qué valores de $\lambda$ falla la unicidad.

5. **Tipo Examen General.** Sea $\Omega\subset\mathbb R^n$ un dominio acotado
   con frontera suave y sea $h\in C(\overline\Omega)$, $h\leq0$. Suponga que

   $$
   \Delta u+h(x)u=f
   $$

   y que existen una subsolución $\underline u$ y una supersolución
   $\overline u$ con

   $$
   \underline u\leq\overline u
   \qquad\text{sobre }\partial\Omega.
   $$

   Demuestre que todo solución clásica $u$ cuyos datos de frontera estén entre
   ambas barreras satisface

   $$
   \underline u\leq u\leq\overline u
   $$

   en $\overline\Omega$.

6. **Tipo Examen General.** Considere

   $$
   \Delta u+c\,u=0
   $$

   en una bola de $\mathbb R^3$, con dato de Dirichlet cero. Analice por qué el
   argumento de unicidad por principio del máximo funciona para $c<0$ y puede
   fallar para $c>0$. Construya un ejemplo radial no trivial para algún $c>0$.

# Control de cobertura y estado del capítulo

## Contenido cubierto

- comparación para funciones armónicas;
- comparación estricta;
- operador $L_h=\Delta+h(x)$;
- definición de subsolución y supersolución;
- principio débil del máximo para $h\leq0$;
- comparación y unicidad para $L_h$;
- método de barreras;
- contraejemplo para $h>0$;
- conexión elemental con valores propios.

## Correcciones y complementos incorporados

- se explicitó la conexidad para la comparación estricta;
- se precisó la convención de signos de subsolución y supersolución;
- se añadió la hipótesis $h\leq0$;
- se completó la demostración mediante una perturbación exponencial;
- se añadió un contraejemplo de resonancia cuando $h>0$.

## Contenido deliberadamente pendiente

No se desarrolla todavía el teorema de diferenciación de Lebesgue que aparece al
final de la página 7.

De acuerdo con la cola del capítulo, el siguiente notebook será

$$
\texttt{03.2.02\_Harnack\_gradiente\_analiticidad.ipynb}.
$$

Allí se trabajará la página 8: regularidad, desigualdad de Harnack,
estimaciones interiores, teorema de Liouville y analiticidad.